# 1회차 · 인공지능 데이터 — Colab 실습
**2026 경기도 공유학교 · 인공지능 기초 (김순찬 교수)** · 성취기준 [인일 02-03] 데이터의 통계적 특성 분석과 시각화로 통찰 얻기

## 오늘의 학습목표
1. 표(DataFrame)의 **행·열·변수 유형**을 구분한다.
2. 평균·중앙값·표준편차·사분위수로 데이터의 **통계적 특성**을 설명한다.
3. 히스토그램·상자그림·산점도·막대·히트맵 중 **목적에 맞는 그래프**를 고른다.
4. 결측·이상치·편향을 찾아내고, 데이터에서 **통찰 3가지**를 글로 정리한다.

## 사용법
- 셀을 클릭한 뒤 **Shift + Enter** → 실행. **반드시 위에서부터 순서대로** 실행하세요(앞 셀에서 만든 변수를 뒤에서 씁니다).
- 오류가 나면 **런타임 → 런타임 다시 시작** 후 처음부터 다시 실행.
- 🧩 표시가 있는 셀은 여러분이 직접 코드를 채우는 셀입니다.

### 셀 1 · 준비 — 도구(라이브러리) 불러오기
- `pandas`는 **표**를 다루는 도구, `numpy`는 **숫자 계산**, `matplotlib`은 **그래프** 도구입니다.
- Colab은 기본 글꼴에 한글이 없어서 그래프의 한글이 □□로 깨집니다. 그래서 나눔 글꼴을 설치합니다(20~30초 걸릴 수 있어요).
- 마지막 줄에 "준비 완료!"가 나오면 성공입니다.

In [ ]:
import matplotlib.pyplot as plt, matplotlib as mpl, numpy as np, pandas as pd
import subprocess, sys
try:
    subprocess.run(['apt-get','-qq','install','-y','fonts-nanum'], check=False, capture_output=True)   # Colab 한글 폰트 설치
    import matplotlib.font_manager as fm; fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf'); plt.rcParams['font.family']='NanumGothic'
except Exception:
    plt.rcParams['font.family'] = 'Noto Sans CJK JP'   # 로컬 검증용
plt.rcParams['axes.unicode_minus'] = False
pd.set_option('display.width', 120)
print('준비 완료! numpy', np.__version__, '/ pandas', pd.__version__)

### 셀 2 · 고등학생 생활 데이터 만들기
- 실제 설문 대신, 컴퓨터가 **가상의 고등학생 200명**의 생활 데이터를 만듭니다(난수 `seed`를 고정했기 때문에 누구나 똑같은 표가 나옵니다).
- 현실 데이터처럼 **빈칸(결측)** 과 **잘못 입력된 값(이상치)** 을 일부러 몇 개 심어 두었습니다. 나중에 여러분이 찾아낼 거예요.
- `df`라는 이름의 **DataFrame(표)** 이 만들어집니다. 이 이름을 앞으로 계속 씁니다.

In [ ]:
def make_student_data(n=200, seed=2026):
    rng = np.random.default_rng(seed)                     # 난수 생성기 (seed 고정 → 누구나 같은 데이터)
    sleep = np.clip(rng.normal(6.5, 1.0, n), 3.5, 9.5)      # 수면시간(시간)
    study = np.clip(rng.normal(2.5, 1.2, n), 0, 7)          # 하루 공부시간(시간)
    phone = np.clip(rng.normal(3.5, 1.3, n), 0.3, 8)        # 스마트폰 사용시간(시간)
    exercise = np.clip(rng.normal(2.0, 1.5, n), 0, 8)       # 주당 운동시간(시간)
    breakfast = rng.choice(['먹음', '안먹음'], n, p=[0.6, 0.4])   # 아침식사 여부
    gender = rng.choice(['남', '여'], n)
    grade = rng.choice([1, 2, 3], n)                        # 학년
    # 성적: 공부시간 ↑, 수면 적당 ↑, 스마트폰 ↑ 이면 ↓ + 잡음
    score = 55 + 6.0 * study - 2.5 * phone + 2.0 * (sleep - 6.5) - 1.5 * (sleep - 7) ** 2 \
            + 3.0 * (breakfast == '먹음') + rng.normal(0, 7, n)
    score = np.clip(score, 20, 100).round(1)
    df = pd.DataFrame({'학년': grade, '성별': gender, '수면시간': sleep.round(1), '공부시간': study.round(1),
                       '스마트폰시간': phone.round(1), '운동시간': exercise.round(1), '아침식사': breakfast, '성적': score})
    # 현실처럼 결측(빈칸)과 이상치(잘못 입력된 값)를 몇 개 심는다
    df.loc[rng.choice(n, 6, replace=False), '수면시간'] = np.nan
    df.loc[rng.choice(n, 4, replace=False), '운동시간'] = np.nan
    df.loc[17, '스마트폰시간'] = 24.0      # 하루 24시간? → 입력 실수
    df.loc[93, '공부시간'] = 15.0         # 하루 15시간 공부? → 의심스러운 값
    df.index.name = '학생번호'
    return df

df = make_student_data()          # 표 만들기
print('표의 크기 (행, 열):', df.shape)
df.head(10)                       # 앞에서 10명만 보기

### 셀 3 · 표의 구조 살펴보기 — `info()`
- 한 **행(row)** = 학생 한 명, 한 **열(column)** = 물어본 항목(변수)입니다.
- `info()`는 열 이름, **비어 있지 않은 값의 개수(Non-Null)**, **자료형(Dtype)** 을 보여 줍니다.
  - `float64`/`int64` → 숫자(수치형), `object` → 글자(범주형).
- 200명인데 Non-Null이 200보다 작은 열이 있다면 → 그 열에 **빈칸**이 있다는 뜻입니다. 어느 열인지 찾아보세요.

In [ ]:
df.info()

### 셀 4 · 숫자 열 한 번에 요약하기 — `describe()`
- `count` 개수 · `mean` 평균 · `std` 표준편차(퍼진 정도) · `min` 최솟값 · `25%/50%/75%` 사분위수(50% = 중앙값) · `max` 최댓값
- 읽는 법: **평균과 중앙값(50%)이 많이 다르면** 한쪽으로 치우쳤거나 이상치가 있다는 신호입니다.
- `max`를 보세요. 하루 스마트폰 24시간? 공부 15시간? **말이 안 되는 값**이 보이나요?

In [ ]:
df.describe().round(2)

### 셀 5 · 빈칸(결측치) 찾기
- `isna()`는 각 칸이 비었는지(True/False) 표시하고, `.sum()`으로 열마다 몇 개인지 셉니다.
- 결측을 그냥 두면 평균 계산 등에서 오류가 나거나 결과가 왜곡됩니다. 처리 방법은 두 가지: **(1) 그 행을 빼기 `dropna()`**, **(2) 대표값으로 채우기 `fillna()`**.
- 여기서는 수면시간·운동시간의 빈칸을 **중앙값**으로 채웁니다(평균보다 이상치의 영향을 덜 받아서).

In [ ]:
print('열마다 빈칸 개수:')
print(df.isna().sum())
print('\n빈칸이 있는 학생 (일부):')
display(df[df.isna().any(axis=1)].head())

df_clean = df.copy()                                   # 원본은 그대로 두고 복사본을 고친다
for col in ['수면시간', '운동시간']:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())   # 중앙값으로 채우기
print('\n채운 뒤 빈칸 개수:', int(df_clean.isna().sum().sum()))

### 셀 6 · 이상치(말이 안 되는 값) 찾기 — 상자그림 규칙
- **IQR 규칙**: Q1(25%)과 Q3(75%) 사이 거리를 IQR이라 하고, `Q1 − 1.5×IQR`보다 작거나 `Q3 + 1.5×IQR`보다 크면 이상치로 봅니다. 상자그림의 ○ 점이 바로 이것입니다.
- 이상치는 **입력 실수**일 수도, **진짜 특별한 학생**일 수도 있습니다. 무조건 지우지 말고 이유를 생각해 보세요. 여기서는 하루 24시간·15시간처럼 **물리적으로 불가능한 값**만 제거합니다.

In [ ]:
def find_outliers(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return s[(s < low) | (s > high)]

for col in ['스마트폰시간', '공부시간', '성적']:
    out = find_outliers(df_clean[col])
    print(f'{col}: 이상치 {len(out)}개 →', out.round(1).to_dict())

# 하루는 24시간! 공부+수면+스마트폰이 24시간을 넘는 학생은 입력 실수로 보고 제거
bad = df_clean[(df_clean['스마트폰시간'] >= 20) | (df_clean['공부시간'] >= 12)]
print('\n제거할 학생:'); display(bad)
df_clean = df_clean.drop(bad.index)
print('정리 후 학생 수:', len(df_clean))

### 셀 7 · 대표값과 퍼짐 — 평균·중앙값·표준편차·사분위수
- **평균(mean)**: 다 더해서 개수로 나눔. 이상치에 잘 끌려감.
- **중앙값(median)**: 크기순으로 세웠을 때 한가운데 값. 이상치에 강함.
- **표준편차(std)**: 평균에서 얼마나 떨어져 있는지의 평균적인 크기. 클수록 넓게 퍼짐.
- **사분위수**: 25%·50%·75% 지점. 상자그림의 상자가 이 범위입니다.
- 정리 **전(df)** 과 **후(df_clean)** 의 스마트폰시간 평균을 비교해 보세요. 이상치 한 명이 평균을 얼마나 바꾸는지 보입니다.

In [ ]:
for name, d in [('정리 전', df), ('정리 후', df_clean)]:
    s = d['스마트폰시간']
    print(f"[{name}] 스마트폰시간  평균 {s.mean():.2f}  중앙값 {s.median():.2f}  표준편차 {s.std():.2f}  최댓값 {s.max():.1f}")

print('\n성적 사분위수:')
print(df_clean['성적'].quantile([0.25, 0.5, 0.75]).round(1))

### 셀 8 · 히스토그램 — 한 변수가 어떻게 퍼져 있나
- 값의 범위를 여러 칸(`bins`)으로 나누고, 각 칸에 학생이 몇 명인지 막대로 그립니다.
- 볼 것: **봉우리가 하나인가 둘인가**, **왼쪽/오른쪽 꼬리가 긴가**, **평균선(빨강 점선)과 중앙값선(초록)이 얼마나 떨어져 있나**.
- `bins` 숫자를 10, 30으로 바꿔 보세요. 칸이 너무 적으면 뭉개지고, 너무 많으면 들쭉날쭉합니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, col in zip(axes, ['성적', '수면시간']):
    s = df_clean[col]
    ax.hist(s, bins=15, color='#4F5BD5', edgecolor='white')
    ax.axvline(s.mean(), color='#C0395F', ls='--', label=f'평균 {s.mean():.1f}')
    ax.axvline(s.median(), color='#0E8C86', label=f'중앙값 {s.median():.1f}')
    ax.set_xlabel(col); ax.set_ylabel('학생 수'); ax.set_title(f'{col} 분포'); ax.legend()
plt.tight_layout(); plt.show()

### 셀 9 · 상자그림(box plot) — 요약통계를 그림 하나로
- 상자 아래쪽 = Q1, 가운데 선 = 중앙값, 위쪽 = Q3. 수염은 정상 범위 끝, 그 밖의 ○는 이상치.
- 여러 변수를 나란히 그리면 **누가 더 넓게 퍼졌는지**, **어디에 이상치가 있는지**가 한눈에 보입니다.
- 학년별 성적을 상자그림으로 비교하면 **그룹 차이**도 볼 수 있습니다(오른쪽 그림).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
cols = ['수면시간', '공부시간', '스마트폰시간', '운동시간']
axes[0].boxplot([df_clean[c] for c in cols], tick_labels=cols, patch_artist=True,
                boxprops=dict(facecolor='#A9DCD8'), medianprops=dict(color='#C0395F', lw=2))
axes[0].set_ylabel('시간'); axes[0].set_title('생활 변수 4개의 상자그림')

groups = [df_clean.loc[df_clean['학년'] == g, '성적'] for g in [1, 2, 3]]
axes[1].boxplot(groups, tick_labels=['1학년', '2학년', '3학년'], patch_artist=True,
                boxprops=dict(facecolor='#C7CAF1'), medianprops=dict(color='#C0395F', lw=2))
axes[1].set_ylabel('성적'); axes[1].set_title('학년별 성적')
plt.tight_layout(); plt.show()

### 셀 10 · 산점도와 상관계수 — 두 변수는 관계가 있나
- 산점도: 학생 한 명 = 점 하나. 점들이 **오른쪽 위로 향하면 양(+)의 관계**, 오른쪽 아래면 **음(−)의 관계**, 동그랗게 퍼지면 관계 없음.
- **상관계수 r**은 −1 ~ +1 사이 숫자. |r|이 0.7 이상이면 강함, 0.3 이하면 약함(대략의 기준).
- ⚠️ 상관 ≠ 인과. "공부시간과 성적이 같이 높다"가 "공부가 성적을 올린다"를 **증명하지는 않습니다**(수업에서 예시 이야기).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, col in zip(axes, ['공부시간', '스마트폰시간', '수면시간']):
    r = df_clean[[col, '성적']].corr().iloc[0, 1]          # 상관계수
    ax.scatter(df_clean[col], df_clean['성적'], s=18, alpha=.7, color='#0E8C86')
    ax.set_xlabel(col + '(시간)'); ax.set_ylabel('성적'); ax.set_title(f'{col} vs 성적   r = {r:+.2f}')
plt.tight_layout(); plt.show()

### 셀 11 · 그룹으로 묶어 비교하기 — `groupby` + 막대그래프
- `groupby('아침식사')['성적'].mean()` = "아침식사 여부별로 묶어서 성적 평균을 구해라".
- 막대그래프는 **범주(그룹)끼리 크기를 비교**할 때 씁니다. 막대 위에 인원(n)을 같이 적으면 더 정직한 그래프가 됩니다.
- 볼 것: 두 그룹 차이가 몇 점인가? 그 차이는 표준편차(약 12점)에 비해 큰가 작은가?

In [ ]:
g = df_clean.groupby('아침식사')['성적'].agg(['mean', 'median', 'count']).round(1)
print(g)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(g.index, g['mean'], color=['#C0395F', '#0E8C86'], width=.5)
for i, (m, c) in enumerate(zip(g['mean'], g['count'])):
    axes[0].text(i, m + 1, f'{m}점 (n={c})', ha='center')
axes[0].set_ylim(0, 100); axes[0].set_ylabel('평균 성적'); axes[0].set_title('아침식사 여부별 평균 성적')

g2 = df_clean.groupby(['학년', '성별'])['성적'].mean().unstack()
g2.plot(kind='bar', ax=axes[1], color=['#4F5BD5', '#D97706'], rot=0)
axes[1].set_ylabel('평균 성적'); axes[1].set_title('학년 × 성별 평균 성적')
plt.tight_layout(); plt.show()

### 셀 12 · 상관계수 히트맵 — 모든 숫자 변수의 관계를 한 장에
- `corr()`는 숫자 열끼리의 상관계수를 **표(행렬)** 로 만들고, 히트맵은 그 표를 **색으로** 보여 줍니다(빨강 = 양, 파랑 = 음).
- 볼 것: 성적 행(맨 아래)에서 **가장 진한 칸**이 무엇인가? 그것이 성적과 가장 관련 있는 변수입니다.
- 대각선은 항상 1(자기 자신과의 상관)이니 무시하세요.

In [ ]:
import seaborn as sns
num_cols = ['수면시간', '공부시간', '스마트폰시간', '운동시간', '성적']
corr = df_clean[num_cols].corr()
print(corr.round(2))
plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1, square=True)
plt.title('상관계수 히트맵'); plt.tight_layout(); plt.show()

### 셀 13 · 🧩 직접 해보기 1 — 운동시간과 성적
`# TODO` 줄을 채워서 **운동시간과 성적의 산점도**를 그리고 상관계수를 제목에 넣어 보세요. (힌트: 셀 10과 거의 같습니다.)

<details><summary>정답 보기</summary>

```python
col = '운동시간'
r = df_clean[[col, '성적']].corr().iloc[0, 1]
plt.scatter(df_clean[col], df_clean['성적'], s=18, color='#D97706')
plt.xlabel(col); plt.ylabel('성적'); plt.title(f'{col} vs 성적  r = {r:+.2f}'); plt.show()
```
</details>

In [ ]:
col = '운동시간'
r = df_clean[[col, '성적']].corr().iloc[0, 1]   # TODO: 상관계수를 구하는 이 줄은 그대로 두고
# TODO: 아래에 plt.scatter( x , y ) 로 산점도를 그리세요
plt.scatter(df_clean[col], df_clean['성적'], s=18, color='#D97706')
plt.xlabel(col); plt.ylabel('성적'); plt.title(f'{col} vs 성적  r = {r:+.2f}'); plt.show()

### 셀 14 · 🧩 직접 해보기 2 — 성별 스마트폰 사용시간
`groupby`를 써서 **성별 스마트폰시간의 평균과 중앙값**을 구하고, 막대그래프로 그려 보세요.

<details><summary>정답 보기</summary>

```python
g = df_clean.groupby('성별')['스마트폰시간'].agg(['mean', 'median']).round(2)
print(g)
g['mean'].plot(kind='bar', color=['#4F5BD5', '#C0395F'], rot=0)
plt.ylabel('평균 스마트폰시간'); plt.show()
```
</details>

In [ ]:
g = df_clean.groupby('성별')['스마트폰시간'].agg(['mean', 'median']).round(2)   # TODO: '스마트폰시간' 열을 골랐는지 확인
print(g)
g['mean'].plot(kind='bar', color=['#4F5BD5', '#C0395F'], rot=0)
plt.ylabel('평균 스마트폰시간'); plt.title('성별 스마트폰 사용시간'); plt.show()

### 셀 15 · (보너스) 진짜 데이터 — 붓꽃(iris)
- `scikit-learn`에 들어 있는 유명한 실제 데이터입니다. 붓꽃 150송이의 꽃잎·꽃받침 길이/너비와 품종(3종).
- 품종별로 색을 다르게 한 산점도를 그리면 **품종이 꽃잎 크기로 갈라지는 패턴**이 보입니다. 다음 시간(지도학습)에 이 패턴으로 품종을 **예측**할 겁니다.

In [ ]:
from sklearn.datasets import load_iris
iris = load_iris(as_frame=True)
X = iris.frame
X['품종'] = [iris.target_names[t] for t in X['target']]
display(X.groupby('품종')[['petal length (cm)', 'petal width (cm)']].mean().round(2))

for name, color in zip(iris.target_names, ['#4F5BD5', '#0E8C86', '#D97706']):
    m = X['품종'] == name
    plt.scatter(X.loc[m, 'petal length (cm)'], X.loc[m, 'petal width (cm)'], s=20, color=color, label=name)
plt.xlabel('꽃잎 길이(cm)'); plt.ylabel('꽃잎 너비(cm)'); plt.title('붓꽃 품종별 꽃잎 크기'); plt.legend(); plt.show()

### 셀 16 · 📝 오늘의 결과물 — 내가 찾은 통찰 3가지
- 위 그래프와 숫자를 근거로 **통찰(insight) 3가지**를 문장으로 적으세요. "무엇이 무엇과 관련 있다 / 어느 그룹이 높다 / 이런 이상치가 있었다" 처럼 **숫자를 포함**해서.
- 예: "공부시간과 성적의 상관계수는 +0.5로, 공부시간이 많을수록 성적이 높은 경향이 있다."
- 실행하면 이름·학번과 함께 요약이 출력됩니다. 화면을 캡처하거나 노트북을 저장(파일 → 다운로드 → .ipynb)해서 제출하세요.

**자기 점검**
- [ ] 결측이 있는 열을 찾고 처리했다  - [ ] 이상치를 찾고 이유를 생각했다
- [ ] 그래프 4종류 이상을 그렸다  - [ ] 통찰 3가지를 숫자와 함께 적었다

In [ ]:
이름 = '홍길동'        # ← 본인 이름
학번 = '10101'        # ← 본인 학번
통찰 = [
    '1. (예) 공부시간과 성적의 상관계수는 +0.xx로 ...',
    '2. ',
    '3. ',
]
print(f'=== 1회차 실습 결과물 · {이름}({학번}) ===')
print(f'분석한 학생 수: {len(df_clean)}명 (원본 {len(df)}명, 이상치 {len(df)-len(df_clean)}명 제거)')
print('성적과의 상관계수:', df_clean[num_cols].corr()['성적'].drop('성적').round(2).to_dict())
print('\n내가 찾은 통찰:')
for t in 통찰: print(' ', t)